In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Users/azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com/sample_supply_chain_data.csv")

In [0]:
print("Original Dataset")
display(df)

Original Dataset


order_id,supplier_id,product_name,order_date,delivery_date,ordered_qty,stock_on_hand,unit_price,status,priority,notes
101,1,Rice,2026-06-01,2026-06-05,50,40,32.5,Delivered,High,On time
102,2,Wheat,2026-06-02,2026-06-08,30,18,28.0,Delivered,Medium,Late due to transport
103,3,Sugar,2026-06-03,null,20,25,41.0,Pending,Low,null
104,1,Salt,2026-06-04,2026-06-06,10,12,15.0,Delivered,High,Fast delivery
105,4,Oil,2026-06-05,2026-06-10,15,8,120.0,Delivered,Medium,Small delay
106,5,Tea,2026-06-06,2026-06-09,40,35,220.0,Delivered,Low,Good
107,2,Coffee,2026-06-07,2026-06-12,25,20,310.0,Delivered,High,Delay
108,3,Milk Powder,2026-06-08,2026-06-13,18,10,450.0,Delivered,Medium,null
109,4,Biscuit,2026-06-09,2026-06-11,60,55,20.0,Delivered,Low,Good stock
110,5,Soap,2026-06-10,null,35,30,25.0,Pending,High,Waiting for shipment


In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- ordered_qty: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- notes: string (nullable = true)



In [0]:
print("Total Records")
df.count()

Total Records


10

In [0]:
df = df.dropDuplicates()

print("After removing duplicates")
display(df)

After removing duplicates


order_id,supplier_id,product_name,order_date,delivery_date,ordered_qty,stock_on_hand,unit_price,status,priority,notes
102,2,Wheat,2026-06-02,2026-06-08,30,18,28.0,Delivered,Medium,Late due to transport
103,3,Sugar,2026-06-03,null,20,25,41.0,Pending,Low,null
106,5,Tea,2026-06-06,2026-06-09,40,35,220.0,Delivered,Low,Good
107,2,Coffee,2026-06-07,2026-06-12,25,20,310.0,Delivered,High,Delay
108,3,Milk Powder,2026-06-08,2026-06-13,18,10,450.0,Delivered,Medium,null
101,1,Rice,2026-06-01,2026-06-05,50,40,32.5,Delivered,High,On time
109,4,Biscuit,2026-06-09,2026-06-11,60,55,20.0,Delivered,Low,Good stock
110,5,Soap,2026-06-10,null,35,30,25.0,Pending,High,Waiting for shipment
104,1,Salt,2026-06-04,2026-06-06,10,12,15.0,Delivered,High,Fast delivery
105,4,Oil,2026-06-05,2026-06-10,15,8,120.0,Delivered,Medium,Small delay


In [0]:
df = df.fillna({
    "status": "Pending",
    "notes": "No Remarks"
})

df.show()

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|     102|          2|       Wheat|2026-06-02|   2026-06-08|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|
|     103|          3|       Sugar|2026-06-03|         NULL|         20|           25|      41.0|  Pending|     Low|          No Remarks|
|     106|          5|         Tea|2026-06-06|   2026-06-09|         40|           35|     220.0|Delivered|     Low|                Good|
|     107|          2|      Coffee|2026-06-07|   2026-06-12|         25|           20|     310.0|Delivered|    High|               Delay|
|     108|          3| Milk Powder

In [0]:
pending = df.filter(col("status") =="Pending")
pending.show()

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+-------+--------+--------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price| status|priority|               notes|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+-------+--------+--------------------+
|     103|          3|       Sugar|2026-06-03|         NULL|         20|           25|      41.0|Pending|     Low|          No Remarks|
|     110|          5|        Soap|2026-06-10|         NULL|         35|           30|      25.0|Pending|    High|Waiting for shipment|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+-------+--------+--------------------+



In [0]:
low_stock = df.filter(
    col("stock_on_hand") < 20
)

low_stock.show()

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|     102|          2|       Wheat|2026-06-02|   2026-06-08|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|
|     108|          3| Milk Powder|2026-06-08|   2026-06-13|         18|           10|     450.0|Delivered|  Medium|          No Remarks|
|     104|          1|        Salt|2026-06-04|   2026-06-06|         10|           12|      15.0|Delivered|    High|       Fast delivery|
|     105|          4|         Oil|2026-06-05|   2026-06-10|         15|            8|     120.0|Delivered|  Medium|         Small delay|
+--------+-----------+------------

In [0]:
supplier_orders = df.groupBy(
    "supplier_id"
).count()

supplier_orders.show()

+-----------+-----+
|supplier_id|count|
+-----------+-----+
|          2|    2|
|          3|    2|
|          5|    2|
|          1|    2|
|          4|    2|
+-----------+-----+



In [0]:
inventory = df.withColumn(
    "inventory_value",
    col("stock_on_hand") * col("unit_price")
)

inventory.groupBy("supplier_id") \
.sum("inventory_value") \
.show()

+-----------+--------------------+
|supplier_id|sum(inventory_value)|
+-----------+--------------------+
|          2|              6704.0|
|          3|              5525.0|
|          5|              8450.0|
|          1|              1480.0|
|          4|              2060.0|
+-----------+--------------------+



In [0]:
status = df.groupBy(
    "status"
).count()

status.show()

+---------+-----+
|   status|count|
+---------+-----+
|Delivered|    8|
|  Pending|    2|
+---------+-----+



In [0]:
df.createOrReplaceTempView("SupplyChain")

In [0]:
%sql
SELECT *
FROM SupplyChain
LIMIT 10;

order_id,supplier_id,product_name,order_date,delivery_date,ordered_qty,stock_on_hand,unit_price,status,priority,notes
102,2,Wheat,2026-06-02,2026-06-08,30,18,28.0,Delivered,Medium,Late due to transport
103,3,Sugar,2026-06-03,null,20,25,41.0,Pending,Low,No Remarks
106,5,Tea,2026-06-06,2026-06-09,40,35,220.0,Delivered,Low,Good
107,2,Coffee,2026-06-07,2026-06-12,25,20,310.0,Delivered,High,Delay
108,3,Milk Powder,2026-06-08,2026-06-13,18,10,450.0,Delivered,Medium,No Remarks
101,1,Rice,2026-06-01,2026-06-05,50,40,32.5,Delivered,High,On time
109,4,Biscuit,2026-06-09,2026-06-11,60,55,20.0,Delivered,Low,Good stock
110,5,Soap,2026-06-10,null,35,30,25.0,Pending,High,Waiting for shipment
104,1,Salt,2026-06-04,2026-06-06,10,12,15.0,Delivered,High,Fast delivery
105,4,Oil,2026-06-05,2026-06-10,15,8,120.0,Delivered,Medium,Small delay


In [0]:
%sql
SELECT supplier_id,COUNT(order_id) AS PendingOrders
FROM SupplyChain
WHERE status='Pending'
GROUP BY supplier_id
ORDER BY PendingOrders DESC;

supplier_id,PendingOrders
3,1
5,1


In [0]:
%sql
SELECT product_name,stock_on_hand,ordered_qty,unit_price
FROM SupplyChain
ORDER BY stock_on_hand;

product_name,stock_on_hand,ordered_qty,unit_price
Oil,8,15,120.0
Milk Powder,10,18,450.0
Salt,12,10,15.0
Wheat,18,30,28.0
Coffee,20,25,310.0
Sugar,25,20,41.0
Soap,30,35,25.0
Tea,35,40,220.0
Rice,40,50,32.5
Biscuit,55,60,20.0


In [0]:
%sql
SELECT supplier_id,COUNT(order_id) AS TotalOrders,
SUM(ordered_qty) AS TotalQuantity,
ROUND(AVG(unit_price),2) AS AveragePrice
FROM SupplyChain
GROUP BY supplier_id;

supplier_id,TotalOrders,TotalQuantity,AveragePrice
2,2,55,169.0
3,2,38,245.5
5,2,75,122.5
1,2,60,23.75
4,2,75,70.0


In [0]:
df.write.mode("overwrite").format("delta").save("dbfs:/delta/SupplyChain")

In [0]:
delta_df = spark.read.format("delta") \
.load("dbfs:/delta/SupplyChain")

delta_df.show()

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|     102|          2|       Wheat|2026-06-02|   2026-06-08|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|
|     103|          3|       Sugar|2026-06-03|         NULL|         20|           25|      41.0|  Pending|     Low|          No Remarks|
|     106|          5|         Tea|2026-06-06|   2026-06-09|         40|           35|     220.0|Delivered|     Low|                Good|
|     107|          2|      Coffee|2026-06-07|   2026-06-12|         25|           20|     310.0|Delivered|    High|               Delay|
|     108|          3| Milk Powder

In [0]:
df.write.mode("overwrite").option("header", True).csv("/FileStore/output/SupplyChainCSV")

In [0]:
csv = spark.read.csv(
    "/FileStore/output/SupplyChainCSV",
    header=True,
    inferSchema=True
)

csv.show()

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+--------------------+
|     102|          2|       Wheat|2026-06-02|   2026-06-08|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|
|     103|          3|       Sugar|2026-06-03|         NULL|         20|           25|      41.0|  Pending|     Low|          No Remarks|
|     106|          5|         Tea|2026-06-06|   2026-06-09|         40|           35|     220.0|Delivered|     Low|                Good|
|     107|          2|      Coffee|2026-06-07|   2026-06-12|         25|           20|     310.0|Delivered|    High|               Delay|
|     108|          3| Milk Powder

In [0]:
df.write.mode("overwrite") \
.format("delta") \
.saveAsTable("SupplyChainTable")

In [0]:
%sql
SELECT * FROM SupplyChainTable;

order_id,supplier_id,product_name,order_date,delivery_date,ordered_qty,stock_on_hand,unit_price,status,priority,notes
102,2,Wheat,2026-06-02,2026-06-08,30,18,28.0,Delivered,Medium,Late due to transport
103,3,Sugar,2026-06-03,null,20,25,41.0,Pending,Low,No Remarks
106,5,Tea,2026-06-06,2026-06-09,40,35,220.0,Delivered,Low,Good
107,2,Coffee,2026-06-07,2026-06-12,25,20,310.0,Delivered,High,Delay
108,3,Milk Powder,2026-06-08,2026-06-13,18,10,450.0,Delivered,Medium,No Remarks
101,1,Rice,2026-06-01,2026-06-05,50,40,32.5,Delivered,High,On time
109,4,Biscuit,2026-06-09,2026-06-11,60,55,20.0,Delivered,Low,Good stock
110,5,Soap,2026-06-10,null,35,30,25.0,Pending,High,Waiting for shipment
104,1,Salt,2026-06-04,2026-06-06,10,12,15.0,Delivered,High,Fast delivery
105,4,Oil,2026-06-05,2026-06-10,15,8,120.0,Delivered,Medium,Small delay


In [0]:
%sql
SELECT product_name,unit_price FROM SupplyChainTable
ORDER BY unit_price DESC
LIMIT 5;

product_name,unit_price
Milk Powder,450.0
Coffee,310.0
Tea,220.0
Oil,120.0
Sugar,41.0


In [0]:
%sql
SELECT product_name,stock_on_hand,ordered_qty
FROM SupplyChainTable
WHERE stock_on_hand < 20;

product_name,stock_on_hand,ordered_qty
Wheat,18,30
Milk Powder,10,18
Salt,12,10
Oil,8,15


In [0]:
print("Azure Databricks ETL Completed Successfully")

Azure Databricks ETL Completed Successfully
